# 01 · Reranker 라우팅
Harness는 reranker로 Chat/Work와 담당 specialist를 고릅니다. 이 기본 실습은 외부 Cohere 호출 대신 점수를 해석할 수 있는 로컬 lexical baseline을 사용합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: Harness Routing Playground</h2>
<p style="margin: 7px 0;">요청을 입력해 fast route와 specialist route의 score, margin, fallback을 확인합니다. 오른쪽에서는 Harness 기본 prompt를 읽기 전용으로 볼 수 있습니다.</p>
  </div>
</div>


In [1]:
from pathlib import Path
import sys
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '5_harness'))
from lab_common.config import load_settings
from src.reranker import FAST_ROUTE_CANDIDATES, SPECIALIST_CANDIDATES, decide
from src.harness_prompts import PROMPT_DEFINITIONS, default_prompt

def route_ui_run(query: str):
    settings = load_settings()
    fast = decide(query, FAST_ROUTE_CANDIDATES, settings)
    specialist = decide(query, SPECIALIST_CANDIDATES, settings)
    return {
        'fast_route': {'selected': fast.selected, 'scores': fast.scores, 'margin': fast.margin, 'fallback': fast.fallback},
        'specialist': {'selected': specialist.selected, 'scores': specialist.scores, 'margin': specialist.margin, 'fallback': specialist.fallback},
    }

with gr.Blocks() as harness_ui:
    gr.Markdown('## Harness Reranker and Prompt Playground')
    with gr.Tab('Routing'):
        query = gr.Textbox(label='Request', value='재고를 분석해서 발주가 필요한 품목을 알려줘.')
        run = gr.Button('Route request', variant='primary')
        result = gr.JSON(label='Routing decision')
        run.click(route_ui_run, inputs=query, outputs=result)
    with gr.Tab('Prompt catalog'):
        prompt_id = gr.Dropdown(choices=list(PROMPT_DEFINITIONS), value='deep_orchestrator', label='Prompt')
        prompt = gr.Textbox(label='Prompt content', lines=18, value=default_prompt('deep_orchestrator'))
        prompt_id.change(default_prompt, inputs=prompt_id, outputs=prompt)
harness_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", height=900)


* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>UI에서 한 번 실행해 결과를 확인한다.</li>
<li>결과가 기대와 다르면 환경 설정과 모델/저장소 이름을 먼저 점검한다.</li>
<li>아래 코드는 같은 기능의 구성 요소를 자세히 살펴보는 선택 심화 자료다.</li>
</ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 사용하는 구성과 동작을 단계별로 확인하는 자료입니다. 기본 실습에서는 실행하지 않아도 됩니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">준비: Harness 라우팅 후보</h2>
<p style="margin: 7px 0;">Harness는 먼저 Chat 또는 Work를 고르고, Work이면 researcher, business, app, writer 같은 specialist 후보를 다시 비교합니다. 여기서는 local lexical baseline으로 흐름을 관찰합니다.</p>
  </div>
</div>


In [ ]:
# Reranker의 핵심: 후보 설명과 요청의 겹치는 단어를 점수로 바꾼 뒤 margin을 확인합니다.
FAST_ROUTES = {"chat": "인사 개념 설명 대화", "work": "조사 분석 재고 발주 문서 생성"}
SPECIALISTS = {"researcher": "근거 시장 문서 조사", "business-agent": "매출 재고 주문 분석 발주", "app-agent": "앱 UI API 코드 배포", "writer": "요약 보고서 메일 작성"}

def rerank(query, candidates):
    words = query.lower().split()
    scores = {name: sum(word.strip(".,?!") in text.lower() for word in words) for name, text in candidates.items()}
    selected = max(scores, key=scores.get); ordered = sorted(scores.values(), reverse=True)
    margin = ordered[0] - ordered[1]
    return {"selected": selected, "scores": scores, "margin": margin, "fallback": margin == 0, "engine": "local_lexical"}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 요청과 후보 점수</h2>
<p style="margin: 7px 0;">선택된 후보뿐 아니라 모든 score와 1위와 2위의 margin을 봅니다. 자동 분기에서는 높은 점수보다 충분한 차이가 있는지가 중요합니다.</p>
  </div>
</div>


In [ ]:
request = "재고를 분석해서 발주가 필요한 품목과 권장 수량을 알려줘."
print("fast route:", rerank(request, FAST_ROUTES))
print("specialist:", rerank(request, SPECIALISTS))


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 모호한 요청의 fallback</h2>
<p style="margin: 7px 0;">margin이 작으면 fallback이 됩니다. 실제 시스템에서는 추가 정보를 묻거나 일반 Chat으로 넘겨 과도한 자동 실행을 피합니다.</p>
  </div>
</div>


In [ ]:
ambiguous = rerank("???", FAST_ROUTES)
print(ambiguous)
assert ambiguous["fallback"]


`fallback=True`는 모델이 애매한 요청을 과도하게 자동 실행하지 않도록 만드는 신호입니다. 실제 Harness의 Cohere reranker를 쓰려면 `.env`에 LiteLLM의 `/rerank` endpoint와 key를 설정하고 이 셀을 다시 실행하세요.